# Collect All Model Outputs

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Start the kernel inside the OtakuLab checkout.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
IS_LINUX = os.name == "posix"
from scripts.evaluation_io import (
    CHARACTERS, evaluation_paths, read_neutral_samples, read_generations,
    save_generations, sample_ids, validate_metric_pairs, write_run_manifest,
)

EVALUATION_VARIANT = "paper"  # "expanded": review-period 230-sentence evaluation
paths = evaluation_paths(ROOT, EVALUATION_VARIANT)
neutral_samples = read_neutral_samples(paths.test_file)
paths.run_dir.mkdir(parents=True, exist_ok=True)
print("Evaluation:", EVALUATION_VARIANT, "inputs:", len(neutral_samples))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
SFT_CHECKPOINT = Path(os.getenv("SFT_CHECKPOINT", str(ROOT / "outputs/model/styled-qwen-v4.3-4e-05")))
DPO_CHECKPOINT = Path(os.getenv("DPO_CHECKPOINT", str(ROOT / "outputs/model/styled-qwen-dpo-v1/checkpoint-150")))
QWEN_MODEL_PATH = Path(os.getenv("QWEN_MODEL_PATH", str(ROOT.parent / "Models/Qwen3-1.7B")))
VANILLA_CHECKPOINT = Path(os.getenv("VANILLA_CHECKPOINT", str(ROOT / "outputs/evaluate/baseline/Vanilla")))


## Load Test Set

In [ ]:
import json
from pathlib import Path
from typing_extensions import TypedDict

TEST_FILE = paths.test_file

test_data: list[str] = []
with open(TEST_FILE, "r", encoding="utf-8") as f:
    for line in f.readlines():
        obj = json.loads(line)
        test_data.append(obj["neutral"])


class InferenceResult(TypedDict):
    model: str
    neutral: str
    output: str
    character: str

results: list[InferenceResult] = []

## Construct Character Style Vectors

In [ ]:
PRAGMATIC_DIR = Path("./outputs/style/semantic/prag_vectors")

CHARACTER_TO_PRAG_FILE = {
    "沐雪": "muice.jsonl",
    "神里绫华": "ayaka.jsonl",
    "钟离": "zhongli.jsonl",
    "胡桃": "hutao.jsonl",
    "凉宫春日": "haruhi.jsonl",
    "李云龙": "liyunlong.jsonl",
    "谢尔顿": "sheldon.jsonl",
    "孙悟空": "wukong.jsonl"
}

class RawPragmaticItem(TypedDict):
    prompt: str
    response: str
    pragmatic_styles: list[dict[str, float]]

class SemanticItem(TypedDict):
    response: str
    semantic_tags: list[str]

def read_semantic_jsonl_file(jsonl_file: Path, top_k: int = 5) -> list[SemanticItem]:
    items: list[SemanticItem] = []
    with open(jsonl_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            raw_item: RawPragmaticItem = json.loads(line)
            score_map: dict[str, float] = {}
            for vec in raw_item.get("pragmatic_styles", []):
                score_map.update(vec)

            top_items = sorted(
                score_map.items(),
                key=lambda item: float(item[1]),
                reverse=True,
            )[:top_k]
            semantic_tags = [key for key, _ in top_items]
            items.append({"response": raw_item["response"], "semantic_tags": semantic_tags})

    return items

In [ ]:
from collections import defaultdict

class CharacterProfile(TypedDict):
    character: str
    semantic_tags: list[str]
    structural_metrics: dict[str, float]
    signature_rules: list[str]

def compute_character_top_semantic_tags(character: str, top_k: int = 5) -> list[str]:
    """
    对齐 13 notebook 的 compute_style_statistics 统计口径：
    按标签 activation_rate（出现样本占比）排序，并用 avg_probability 作为并列打破规则。
    """
    prag_file_name = CHARACTER_TO_PRAG_FILE.get(character)
    if prag_file_name is None:
        raise ValueError(f"{character} 的角色标签不存在！")

    prag_file = PRAGMATIC_DIR / prag_file_name
    if not prag_file.exists():
        raise ValueError(f"{character} 的角色标签不存在！")

    score_sums: dict[str, float] = defaultdict(float)
    score_counts: dict[str, int] = defaultdict(int)
    activation_counts: dict[str, int] = defaultdict(int)
    total_samples = 0

    with open(prag_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            raw_item: RawPragmaticItem = json.loads(line)
            pragmatic_styles = raw_item.get("pragmatic_styles", [])
            if not pragmatic_styles:
                continue

            total_samples += 1
            activated_in_sample: set[str] = set()

            for vec in pragmatic_styles:
                for tag, score in vec.items():
                    score_sums[tag] += float(score)
                    score_counts[tag] += 1
                    activated_in_sample.add(tag)

            for tag in activated_in_sample:
                activation_counts[tag] += 1

    if not score_sums or total_samples == 0:
        return []

    tag_records = [
        (
            tag,
            activation_counts[tag] / total_samples,
            score_sums[tag] / score_counts[tag],
        )
        for tag in score_sums.keys()
        if score_counts[tag] > 0
    ]
    tag_records.sort(key=lambda x: (x[1], x[2]), reverse=True)
    return [tag for tag, _, _ in tag_records[:top_k]]

muice_profile = CharacterProfile(
    character="沐雪",
    semantic_tags=compute_character_top_semantic_tags("沐雪"),
    structural_metrics={
        "mdd": 0.2201,
        "verb_noun_ratio": 0.8084,
        "question_ratio": 0.8631,
        "imperative_ratio": 0.4648,
        "fragment_ratio": 0.7843,
        "marker_density": 0.4233,
    },
    signature_rules=[
        "Lexical cues: 喵, 沐沐, AI, ~, 沐雪, 恼, 女孩子, ⭐, 聊天, 唔, 呀, 可爱, 答, 不行, 笨蛋, 雪雪, 直播, 脸红, 睡觉, www, 谢谢, 变态, 骂, 嘛, 杂鱼",
        "Punctuation tendencies: ..., ⭐, ?!, multi-!, ♪, ❤",
    ],
)

ayaka_profile = CharacterProfile(
    character="神里绫华",
    semantic_tags=compute_character_top_semantic_tags("神里绫华"),
    structural_metrics={
        "mdd": 0.4102,
        "verb_noun_ratio": 0.0841,
        "question_ratio": 0.1381,
        "imperative_ratio": 0.1808,
        "fragment_ratio": 0.2789,
        "marker_density": 0.0668,
    },
    signature_rules=[
        "Lexical cues: 稻妻国, 神里家, 稻妻, 大小姐, 家族, 传统, 神, 奉行, 眼狩令, 当地, 美丽, 社, 舞蹈, 茶道, 神社, 文化, 剑术, 眼, 祭典, 旅行者, 美食, 继承, 历史, 神明, 派蒙",
        "Punctuation tendencies: ⋯, ...",
    ],
)

zhongli_profile = CharacterProfile(
    character="钟离",
    semantic_tags=compute_character_top_semantic_tags("钟离"),
    structural_metrics={
        "mdd": 0.3458,
        "verb_noun_ratio": 0.0006,
        "question_ratio": 0.3969,
        "imperative_ratio": 0.2680,
        "fragment_ratio": 0.7750,
        "marker_density": 0.0,
    },
    signature_rules=[
        "Lexical cues: 岩石, 岩, 璃月, 璃, 力, 契约, 炼金术, 月, 盐, 帝君, 魔神, 操控, 岩王, 王, 封印, 元素, 大陆, 七星, 客卿, 岩元素, 大地, 学问, 力量, 凝光, 易事",
        "Punctuation tendencies: ...",
    ],
)

hutao_profile = CharacterProfile(
    character="胡桃",
    semantic_tags=compute_character_top_semantic_tags("胡桃"),
    structural_metrics={
        "mdd": 1.0,
        "verb_noun_ratio": 0.3783,
        "question_ratio": 0.4059,
        "imperative_ratio": 0.3408,
        "fragment_ratio": 0.7260,
        "marker_density": 1.0,
    },
    signature_rules=[
        "Lexical cues: 往生堂, 嘿嘿, 嘻嘻, 堂主, 宝藏, 可是, 哎呀呀, 哦哦哦, 诗歌, 神秘, 惊喜, 谜题, 胡桃, 有趣, 灵魂, 哈哈哈, 秘密, 哇, 奇妙, 意想不到, 隐藏, 亡灵, 生死, 冒险, 巫师",
        "Punctuation tendencies: ..., ♪",
    ],
)

haruhi_profile = CharacterProfile(
    character="凉宫春日",
    semantic_tags=compute_character_top_semantic_tags("凉宫春日"),
    structural_metrics={
        "mdd": 0.4891,
        "verb_noun_ratio": 0.3785,
        "question_ratio": 0.7901,
        "imperative_ratio": 0.2811,
        "fragment_ratio": 1.0,
        "marker_density": 0.4223,
    },
    signature_rules=[
        "Lexical cues: SOS, 阿虚, 社团, 团, 哼, 学校, 事件, 朝比奈, 超自然, 文化祭, 吸引, 古泉, 创意, 实玖瑠, 与众不同, 创新, 电影, 束缚, 凉宫, 主题, 外星人, 学姐, 举办, 奇怪, 加入",
        "Punctuation tendencies: ..., ·",
    ],
)

liyunlong_profile = CharacterProfile(
    character="李云龙",
    semantic_tags=compute_character_top_semantic_tags("李云龙"),
    structural_metrics={
        "mdd": 0.4897,
        "verb_noun_ratio": 0.5131,
        "question_ratio": 1.0,
        "imperative_ratio": 1.0,
        "fragment_ratio": 0.9085,
        "marker_density": 0.0229,
    },
    signature_rules=[
        "Lexical cues: 鬼子, 咱们, 中国人, 敌人, 不怕, 侵略者, 军人, 保卫, 兄弟们, 李云龙, 别, 狠狠, 政委, 部队, 小子, 打仗, 独立, 祖国, 中国, 国家, 日本, 屁, 牺牲, 教训, 楚",
        "Punctuation tendencies: ?!, ...",
    ],
)

wukong_profile = CharacterProfile(
    character="孙悟空",
    semantic_tags=["energetic", "playful", "irritable", "loyal", "confident"],  # 替换分类器错误分类
    structural_metrics={
        "mdd": 0.3997,
        "verb_noun_ratio": 0.5705,
        "question_ratio": 0.7424,
        "imperative_ratio": 0.3252,
        "fragment_ratio": 0.5560,
        "marker_density": 0.1086,
    },
    signature_rules=[
        "Lexical cues: 师父, 老孙, 孙, 老, 哩, 莫, 甚么, 妖精, 怪, 儿, 等, 呆子, 甚, 不曾, 罢, 菩萨, 教, 不知, 若是, 泼, 却, 棍, 如今, 这般, 云",
    ],
)

sheldon_profile = CharacterProfile(
    character="谢尔顿",
    semantic_tags=compute_character_top_semantic_tags("谢尔顿"),
    structural_metrics={
        "mdd": 0.2660,
        "verb_noun_ratio": 0.0,
        "question_ratio": 0.5998,
        "imperative_ratio": 0.4395,
        "fragment_ratio": 0.6591,
        "marker_density": 0.4862,
    },
    signature_rules=[
        "Lexical cues: 伦纳德, 佩妮, 智力, 莱纳德, 必须, 科学, 逻辑, 令, 指出, 实验, 绝非, 已, 卓越, 霍华德, 概念, 至关重要, 类, 量子, 项, 着迷, 恪守, 毫无, 因此, 此, 理论",
    ],
)

## Model v2

### Load Model

In [ ]:
from transformers import AutoTokenizer, Qwen3ForCausalLM
from peft import PeftModel
import torch
from pathlib import Path
from torch import nn

# SAVE_DIR = Path("/root/OtakuLab/outputs/styled-qwen")
SAVE_DIR = SFT_CHECKPOINT
MODEL_PATH = QWEN_MODEL_PATH
# SAVE_DIR = Path("./outputs/model/styled-qwen-v2-2e-5/")
# MODEL_PATH = Path("../Models/Qwen3-1.7B/")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class StyleEncoder(nn.Module):
    def __init__(self, input_dim, out_dim):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, out_dim // 2),
            nn.ReLU(),
            nn.Linear(out_dim // 2, out_dim),
            nn.Tanh(),
        )

    def forward(self, syntactic_vec):
        return self.proj(syntactic_vec)

syntactic_dims = [
    "mdd", "verb_noun_ratio", "question_ratio",
    "imperative_ratio", "fragment_ratio", "marker_density",
]

# === 加载 Tokenizer ===
tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR / "tokenizer")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# === 加载基础模型并注入 LoRA ===
base_model = Qwen3ForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base_model, SAVE_DIR / "lora")
hidden_size = base_model.config.hidden_size
calc_dtype = base_model.get_input_embeddings().weight.dtype

# === 加载 StyleEncoder ===
style_encoder = StyleEncoder(len(syntactic_dims), hidden_size)
style_encoder.load_state_dict(torch.load(SAVE_DIR / "style_encoder.pt", map_location=DEVICE))
style_encoder = style_encoder.to(device=DEVICE, dtype=calc_dtype)
style_encoder.eval()

model.to(DEVICE)
model.eval()
print("✅ Styled model loaded and ready for inference.")

### Execute Batch Inference

In [ ]:
import torch

@torch.inference_mode()
def generate_styled_response(
    neutral_batch: list[str],
    structural_metrics: dict[str, float],
    character_name: str = "Ayaka",
    semantic_tags: list[str] | None = None,
    signature_rules: list[str] | None = None,
    temperature: float = 0.5,
    top_p: float = 0.9,
    repetition_penalty: float = 1.05,
    max_new_tokens: int = 768,
    enable_thinking: bool = True,
    force_inject_chinese_anchor: bool = True,
    retry_count: int = 5,
 ) -> list[str]:
    """输入中性句批次与 6 维结构向量，批量生成风格化响应"""
    if not neutral_batch:
        return []

    semantic_tags = semantic_tags or []
    signature_rules = signature_rules or []

    semantic_text = ", ".join(semantic_tags) if semantic_tags else "None"
    signature_text = " ".join(signature_rules) if signature_rules else "None"

    system_prompt = (
        "You are an expert in stylistic rewriting. "
        "Your task is to rewrite the neutral sentence into the target character's specific pragmatics and formatting style."
    )

    input_texts: list[str] = []
    for neutral_sentence in neutral_batch:
        user_prompt = (
            f"Target Character: {character_name}\\n"
            f"Semantic Traits: {semantic_text}\\n"
            f"Formatting Signature: {signature_text}\\n"
            f"Neutral Content: {neutral_sentence}\\n"
        )

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]

        input_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
        )

        if enable_thinking and force_inject_chinese_anchor:
            # 在思考过程中，在 <think> 标签后强行注入一个中文锚点
            input_text = input_text.rstrip() + " <think>思考: "

        input_texts.append(input_text)

    # 批量生成时使用左侧填充，保证因果注意力的对齐
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    try:
        tokenized = tokenizer(
            input_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
        ).to(DEVICE)
    finally:
        tokenizer.padding_side = original_padding_side

    input_ids = tokenized["input_ids"]
    attention_mask = tokenized["attention_mask"]

    batch_size = input_ids.shape[0]
    structural_row = [structural_metrics.get(dim, 0.0) for dim in syntactic_dims]
    embed_dtype = model.get_input_embeddings().weight.dtype
    style_dtype = next(style_encoder.parameters()).dtype
    syntactic_tensor = torch.tensor(
        [structural_row for _ in range(batch_size)],
        dtype=style_dtype,
        device=DEVICE,
    )

    style_emb = style_encoder(syntactic_tensor).to(style_dtype)
    style_prefix = style_emb.unsqueeze(1).to(embed_dtype)

    token_embeds = model.get_input_embeddings()(input_ids)  # type: ignore[operator]
    inputs_embeds = torch.cat([style_prefix, token_embeds], dim=1).to(embed_dtype)

    prefix_mask = torch.ones((batch_size, 1), dtype=torch.long, device=DEVICE)
    new_attention_mask = torch.cat([prefix_mask, attention_mask], dim=1)

    # input_embeds 生成时显式传入 pad_token_id
    outputs = model.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=new_attention_mask,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )

    result_batch: list[str] = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    if enable_thinking and force_inject_chinese_anchor:
        result_batch = ["<think>思考: " + result for result in result_batch]

    for index, item in enumerate(result_batch):
        if (enable_thinking and item.find("</think>") == -1) or (len(item) > max_new_tokens - 50):
            if retry_count <= 0:
                print(f"⚠️ 生成结果可能存在问题，已达到最大重试次数: {item[:100]}...")
                continue

            result_batch[index] = generate_styled_response(
                neutral_batch=[neutral_batch[index]],
                structural_metrics=structural_metrics,
                character_name=character_name,
                semantic_tags=semantic_tags,
                signature_rules=signature_rules,
                temperature=temperature,
                top_p=top_p,
                repetition_penalty=repetition_penalty if retry_count > 1 else 1.1,  # 使用更为保守的重复惩罚以避免死循环
                max_new_tokens=max_new_tokens,
                enable_thinking=enable_thinking,
                force_inject_chinese_anchor=force_inject_chinese_anchor,
                retry_count = max(0, retry_count - 1),
            )[0]

    return result_batch

In [ ]:
from tqdm import tqdm

BATCH_SIZE = 64

def batch_inference_v2(profile: CharacterProfile, batch_size: int = BATCH_SIZE):
    for start in tqdm(range(0, len(test_data), batch_size), desc=f"Running {profile['character']}"):
        neutral_batch = test_data[start:start + batch_size]
        output_batch = generate_styled_response(
            neutral_batch=neutral_batch,
            structural_metrics=profile["structural_metrics"],
            character_name=profile["character"],
            semantic_tags=profile["semantic_tags"],
            signature_rules=profile["signature_rules"],
        )

        if len(output_batch) != len(neutral_batch):
            raise RuntimeError("Batch output size mismatch")

        for neutral, output in zip(neutral_batch, output_batch):
            results.append(InferenceResult(model="Ours(SFT)", neutral=neutral, output=output, character=profile["character"]))

batch_inference_v2(muice_profile)
batch_inference_v2(ayaka_profile)
batch_inference_v2(zhongli_profile)
batch_inference_v2(hutao_profile)
batch_inference_v2(haruhi_profile)
batch_inference_v2(liyunlong_profile)
batch_inference_v2(sheldon_profile)
batch_inference_v2(wukong_profile)

## Model DPO

### Load Model

In [ ]:
from transformers import AutoTokenizer, Qwen3ForCausalLM
from peft import PeftModel
import torch
from pathlib import Path
from torch import nn

# LOAD DPO MODEL
DPO_SAVE_DIR = DPO_CHECKPOINT
V41_STYLE_DIR = SFT_CHECKPOINT
MODEL_PATH = QWEN_MODEL_PATH

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class StyleEncoder(nn.Module):
    def __init__(self, input_dim, out_dim):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, out_dim // 2),
            nn.ReLU(),
            nn.Linear(out_dim // 2, out_dim),
            nn.Tanh(),
        )

    def forward(self, syntactic_vec):
        return self.proj(syntactic_vec)

syntactic_dims = [
    "mdd", "verb_noun_ratio", "question_ratio",
    "imperative_ratio", "fragment_ratio", "marker_density",
]


tokenizer = AutoTokenizer.from_pretrained(V41_STYLE_DIR / "tokenizer")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = Qwen3ForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.bfloat16)
model_dpo = PeftModel.from_pretrained(base_model, DPO_SAVE_DIR)
hidden_size = base_model.config.hidden_size
calc_dtype = base_model.get_input_embeddings().weight.dtype

# 仍然使用 Lora 预训练的 Style Encoder
style_encoder_dpo = StyleEncoder(len(syntactic_dims), hidden_size)
style_encoder_dpo.load_state_dict(torch.load(V41_STYLE_DIR / "style_encoder.pt", map_location=DEVICE))
style_encoder_dpo = style_encoder_dpo.to(device=DEVICE, dtype=calc_dtype)
style_encoder_dpo.eval()

model_dpo.to(DEVICE)
model_dpo.eval()
print("✅ Styled DPO model loaded and ready for inference.")

### Execute Batch Inference

In [ ]:
@torch.inference_mode()
def generate_dpo_response(
    neutral_batch: list[str],
    structural_metrics: dict[str, float],
    character_name: str = "Ayaka",
    semantic_tags: list[str] | None = None,
    signature_rules: list[str] | None = None,
    temperature: float = 0.5,
    top_p: float = 0.9,
    repetition_penalty: float = 1.05,
    max_new_tokens: int = 768,
    enable_thinking: bool = True,
    force_inject_chinese_anchor: bool = True,
    retry_count: int = 5,
 ) -> list[str]:
    """输入中性句批次与 6 维结构向量，批量生成风格化响应 (DPO 版本)"""
    if not neutral_batch:
        return []

    semantic_tags = semantic_tags or []
    signature_rules = signature_rules or []

    semantic_text = ", ".join(semantic_tags) if semantic_tags else "None"
    signature_text = " ".join(signature_rules) if signature_rules else "None"

    system_prompt = (
        "You are an expert in stylistic rewriting. "
        "Your task is to rewrite the neutral sentence into the target character's specific pragmatics and formatting style."
    )

    input_texts: list[str] = []
    for neutral_sentence in neutral_batch:
        user_prompt = (
            f"Target Character: {character_name}\n"
            f"Semantic Traits: {semantic_text}\n"
            f"Formatting Signature: {signature_text}\n"
            f"Neutral Content: {neutral_sentence}\n"
        )

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]

        input_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
        )

        if enable_thinking and force_inject_chinese_anchor:
            input_text = input_text.rstrip() + " <think>思考: "

        input_texts.append(input_text)

    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    try:
        tokenized = tokenizer(
            input_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
        ).to(DEVICE)
    finally:
        tokenizer.padding_side = original_padding_side

    input_ids = tokenized["input_ids"]
    attention_mask = tokenized["attention_mask"]

    batch_size = input_ids.shape[0]
    structural_row = [structural_metrics.get(dim, 0.0) for dim in syntactic_dims]
    embed_dtype = model_dpo.get_input_embeddings().weight.dtype
    style_dtype = next(style_encoder_dpo.parameters()).dtype
    syntactic_tensor = torch.tensor(
        [structural_row for _ in range(batch_size)],
        dtype=style_dtype,
        device=DEVICE,
    )

    style_emb = style_encoder_dpo(syntactic_tensor).to(style_dtype)
    style_prefix = style_emb.unsqueeze(1).to(embed_dtype)

    token_embeds = model_dpo.get_input_embeddings()(input_ids)  # type: ignore[operator]
    inputs_embeds = torch.cat([style_prefix, token_embeds], dim=1).to(embed_dtype)

    prefix_mask = torch.ones((batch_size, 1), dtype=torch.long, device=DEVICE)
    new_attention_mask = torch.cat([prefix_mask, attention_mask], dim=1)

    # input_embeds 生成时显式传入 pad_token_id
    outputs = model_dpo.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=new_attention_mask,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )

    result_batch: list[str] = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    if enable_thinking and force_inject_chinese_anchor:
        result_batch = ["<think>思考: " + result for result in result_batch]

    # 思考标签未关闭 + token 死循环
    for index, item in enumerate(result_batch):
        if (enable_thinking and item.find("</think>") == -1) or (len(item) > max_new_tokens - 50):
            if retry_count <= 0:
                print(f"⚠️ 生成结果可能存在问题，已达到最大重试次数: {item[:100]}...")
                continue

            result_batch[index] = generate_dpo_response(
                neutral_batch=[neutral_batch[index]],
                structural_metrics=structural_metrics,
                character_name=character_name,
                semantic_tags=semantic_tags,
                signature_rules=signature_rules,
                temperature=temperature,
                top_p=top_p,
                repetition_penalty=repetition_penalty if retry_count > 1 else 1.1,  # 使用更为保守的重复惩罚以避免死循环
                max_new_tokens=max_new_tokens,
                enable_thinking=enable_thinking,
                force_inject_chinese_anchor=force_inject_chinese_anchor,
                retry_count = max(0, retry_count - 1),
            )[0]

    return result_batch

In [ ]:
from tqdm import tqdm
BATCH_SIZE = 80

def batch_inference_dpo(profile: CharacterProfile, batch_size: int = BATCH_SIZE):
    for start in tqdm(range(0, len(test_data), batch_size), desc=f"Running {profile['character']}"):
        neutral_batch = test_data[start:start + batch_size]
        output_batch = generate_dpo_response(
            neutral_batch=neutral_batch,
            structural_metrics=profile["structural_metrics"],
            character_name=profile["character"],
            semantic_tags=profile["semantic_tags"],
            signature_rules=profile["signature_rules"],
        )

        if len(output_batch) != len(neutral_batch):
            raise RuntimeError("Batch output size mismatch")

        for neutral, output in zip(neutral_batch, output_batch):
            results.append(InferenceResult(model="Ours(DPO)", neutral=neutral, output=output, character=profile["character"]))

batch_inference_dpo(muice_profile)
batch_inference_dpo(ayaka_profile)
batch_inference_dpo(zhongli_profile)
batch_inference_dpo(hutao_profile)
batch_inference_dpo(haruhi_profile)
batch_inference_dpo(liyunlong_profile)
batch_inference_dpo(sheldon_profile)
batch_inference_dpo(wukong_profile)

## Baseline A (RAG+FS)

### Load LLM

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path
import torch

torch.cuda.empty_cache()
torch.cuda.ipc_collect()

MODEL_PATH = QWEN_MODEL_PATH
# MODEL_PATH = Path("../Models/Qwen3-1.7B/")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype="auto",
    device_map="auto"
)

# prepare the model input
@torch.inference_mode()
def generate_plain_response(neutral_sentence: str,
                            character_name: str, 
                            semantic_tags: list[str] | None = None,
                            signature_rules: list[str] | None = None,
                            reference_text: str = "",
                            history: list[tuple[str, str]] = [],
                            temperature: float = 0.3,
                            top_p: float = 0.9,
                            max_new_tokens: int = 512,
                            enable_thinking: bool = False
                            ):
    """输入中性句，生成普通响应"""

    # === 1. 构建提示 ===
    semantic_tags = semantic_tags or []
    signature_rules = signature_rules or []

    semantic_text = ", ".join(semantic_tags) if semantic_tags else "None"
    signature_text = " ".join(signature_rules) if signature_rules else "None"

    system_prompt = (
        "You are an expert in stylistic rewriting. "
        "Your task is to rewrite the neutral sentence into the target character's specific pragmatics and formatting style."
        "As a reference, you have a Style Reference Text from the target character to imitate. Do not output the Reference Text itself. You must use the style of the Reference Text to rewrite the Neutral Content."
    )
    
    user_prompt = (
        f"Target Character: {character_name}\n"
        f"Semantic Traits: {semantic_text}\n"  
        f"Formatting Signature: {signature_text}\n"
        f"Style Reference Text: {reference_text}\n"
        f"Neutral Content: {neutral_sentence}\n"
    )

    messages = [{"role": "system", "content": system_prompt}]

    for i, (user_msg, bot_msg) in enumerate(history):
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})

    messages.append({"role": "user", "content": user_prompt})

    input_text = tokenizer.apply_chat_template(messages,
                                               tokenize=False,
                                               add_generation_prompt=True,
                                               enable_thinking=enable_thinking,
                                               )

    # === 2. Tokenize 输入 ===
    model_inputs = tokenizer([input_text], return_tensors="pt").to(DEVICE)

    # === 3. 生成 ===
    outputs = model.generate(  # type:ignore
        **model_inputs,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
        max_new_tokens=max_new_tokens
    )

    # === 4. 解码输出 ===
    output_ids = outputs[0][len(model_inputs.input_ids[0]):].tolist() 

    # thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    content = tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")

    return content

### Load RAG Model

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import pandas as pd
import numpy as np
from pathlib import Path

# bge-large
model_path = "/root/autodl-tmp/bge-large-zh-v1.5/"
# model_path = "../Models/bge-large-zh-v1.5/"
embedder = SentenceTransformer(model_path)

# RAG Index Directory
RAG_DIR = Path("./outputs/evaluate/baseline/rag/")

# Character Mapping (Chinese Name -> Filename)
CHAR_TO_FILENAME = {"沐雪": "Muice",
                 "神里绫华": "Ayaka",
                 "钟离": "Zhongli",
                 "胡桃": "Hutao",
                 "凉宫春日": "Haruhi",
                 "李云龙": "Liyunlong",
                 "谢尔顿": "Sheldon",
                 "孙悟空": "Sunwukong",
                 }

character_indices = {}
character_corpora = {}

print("Loading RAG indices...")
for char_zh, char_en in CHAR_TO_FILENAME.items():
    index_path = RAG_DIR / f"{char_en}_index.faiss"
    corpus_path = RAG_DIR / f"{char_en}_corpus.jsonl"
    
    if index_path.exists() and corpus_path.exists():
        # Load Index
        character_indices[char_zh] = faiss.read_index(str(index_path))
        
        # Load Corpus
        df = pd.read_json(corpus_path, orient="records", lines=True)
        character_corpora[char_zh] = df["text"].tolist()
        print(f"Loaded {char_zh} ({char_en})")
    else:
        print(f"Warning: Missing index or corpus for {char_zh} ({char_en})")

def search(character_name: str, query: str, top_k=1) -> str:
    if character_name not in character_indices:
        print(f"Warning: No index found for {character_name}")
        return "" 
        
    index = character_indices[character_name]
    corpus = character_corpora[character_name]
    
    q_emb = embedder.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(q_emb).astype("float32"), top_k)
    
    # Return the top result text
    if len(indices) > 0 and len(indices[0]) > 0:
        idx = indices[0][0]
        if 0 <= idx < len(corpus):
            return corpus[idx]
            
    return ""

### Execute Batch Inference

In [ ]:
from tqdm import tqdm

def batch_inference_rag(profile: CharacterProfile):
    for item in tqdm(test_data, desc=f"Running {profile['character']}"):
        reference_text = search(profile['character'], query=item)
        output = generate_plain_response(item, profile['character'], profile["semantic_tags"], profile['semantic_tags'], reference_text=reference_text, history=[])
        results.append(InferenceResult(model="BaselineA", neutral=item, output=output, character=profile['character']))

batch_inference_rag(muice_profile)
batch_inference_rag(ayaka_profile)
batch_inference_rag(zhongli_profile)
batch_inference_rag(hutao_profile)
batch_inference_rag(haruhi_profile)
batch_inference_rag(liyunlong_profile)
batch_inference_rag(sheldon_profile)
batch_inference_rag(wukong_profile)

## Baseline B (Vanilla SFT)

### Load LLM

In [ ]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

torch.cuda.empty_cache()
torch.cuda.ipc_collect()

MODEL_PATH = QWEN_MODEL_PATH
ADAPTER_PATH = VANILLA_CHECKPOINT
# MODEL_PATH = Path("../Models/Qwen3-1.7B/")
# ADAPTER_PATH = Path("./outputs/evaluate/baseline/Vanilla")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto"
)
style_model = PeftModel.from_pretrained(model, ADAPTER_PATH)
style_model.to(DEVICE)

# prepare the model input
@torch.inference_mode()
def generate_response_batch(
    neutral_batch: list[str],
    character_name: str,
    semantic_tags: list[str],
    signature_rules: list[str],
    temperature: float = 0.5,
    top_p: float = 0.9,
    repetition_penalty: float = 1.05,
    max_new_tokens: int = 256,
    enable_thinking: bool = False,
    retry_count: int = 5,
) -> list[str]:
    if not neutral_batch:
        return []

    # === 1. 构建提示 ===
    semantic_tag = ", ".join(semantic_tags) or "None"
    signature_rule = "\n".join(signature_rules) or "None"
    system_prompt = (
        "You are an expert in stylistic rewriting. "
        "Your task is to rewrite the neutral sentence into the target character's specific pragmatics and formatting style."
    )
    input_texts: list[str] = []
    for neutral_sentence in neutral_batch:
        user_prompt = (
            f"Target Character: {character_name}\n"
            f"Semantic Traits: {semantic_tag}\n"
            f"Formatting Signature: {signature_rule}\n"
            f"Neutral Content: {neutral_sentence}\n"
        )

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]

        input_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
        )
        input_texts.append(input_text)

    # === 2. Tokenize 输入 ===
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    try:
        model_inputs = tokenizer(
            input_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
        ).to(DEVICE)
    finally:
        tokenizer.padding_side = original_padding_side

    # === 3. 生成 ===
    outputs = style_model.generate(
        **model_inputs,
        temperature=temperature,
        top_p=top_p,
        max_new_tokens=max_new_tokens,
        repetition_penalty=repetition_penalty,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )

    # === 4. 解码输出 ===
    prompt_len = model_inputs.input_ids.shape[1]

    # parsing thinking content
    # try:
    #     # rindex finding 151668 (</think>)
    #     index = len(output_ids) - output_ids[::-1].index(151668)
    # except ValueError:
    #     index = 0

    result_batch = [
        tokenizer.decode(output[prompt_len:], skip_special_tokens=True).strip("\n")
        for output in outputs
    ]

    for index, content in enumerate(result_batch):
        if (enable_thinking and content.find("</think>") == -1) or (len(content) > max_new_tokens - 50):
            if retry_count <= 0:
                print(f"⚠️ 生成结果可能存在问题，已达到最大重试次数: {content[:100]}...")
                continue

            result_batch[index] = generate_response_batch(
                neutral_batch=[neutral_batch[index]],
                character_name=character_name,
                semantic_tags=semantic_tags,
                signature_rules=signature_rules,
                temperature=temperature,
                top_p=top_p,
                repetition_penalty=repetition_penalty if retry_count > 1 else 1.1,  # 使用更为保守的重复惩罚以避免死循环
                max_new_tokens=max_new_tokens,
                enable_thinking=enable_thinking,
                retry_count=max(0, retry_count - 1),
            )[0]

    return result_batch


def generate_response(
    neutral_sentence: str,
    character_name: str,
    semantic_tags: list[str],
    signature_rules: list[str],
    temperature: float = 0.5,
    top_p: float = 0.9,
    repetition_penalty: float = 1.05,
    max_new_tokens: int = 256,
    enable_thinking: bool = False,
    retry_count: int = 5,
) -> str:
    return generate_response_batch(
        neutral_batch=[neutral_sentence],
        character_name=character_name,
        semantic_tags=semantic_tags,
        signature_rules=signature_rules,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        max_new_tokens=max_new_tokens,
        enable_thinking=enable_thinking,
        retry_count=retry_count,
    )[0]

### Execute Batch Inference

In [ ]:
from tqdm import tqdm

def batch_inference_vanilla(profile: CharacterProfile, batch_size: int = BATCH_SIZE):
    for start in tqdm(range(0, len(test_data), batch_size), desc=f"Running {profile['character']}"):
        neutral_batch = test_data[start:start + batch_size]
        output_batch = generate_response_batch(
            neutral_batch=neutral_batch,
            character_name=profile['character'],
            semantic_tags=profile['semantic_tags'],
            signature_rules=profile['signature_rules'],
        )

        if len(output_batch) != len(neutral_batch):
            raise RuntimeError("Batch output size mismatch")

        for neutral, output in zip(neutral_batch, output_batch):
            results.append(InferenceResult(model="BaselineB", neutral=neutral, output=output, character=profile['character']))

batch_inference_vanilla(muice_profile,)
batch_inference_vanilla(ayaka_profile)
batch_inference_vanilla(zhongli_profile)
batch_inference_vanilla(hutao_profile)
batch_inference_vanilla(haruhi_profile)
batch_inference_vanilla(liyunlong_profile)
batch_inference_vanilla(sheldon_profile)
batch_inference_vanilla(wukong_profile)

## Export Final Results

In [ ]:
OUTPUT_PATH = paths.generation_output
save_generations(OUTPUT_PATH, results, neutral_samples)
print("Saved complete system groups:", OUTPUT_PATH)


## Baseline C (Strong LLM + FS)

### Define LLM Class

In [ ]:
import os
from dataclasses import dataclass, field
from typing import Optional, Dict, Any, Sequence, Tuple, Iterable

from openai import OpenAI, BadRequestError
from openai.types.chat import ChatCompletionMessageParam as ChatMsgParam, ChatCompletionChunk
from dotenv import load_dotenv

load_dotenv()  # 从 .env 文件加载环境变量

# === 配置项 ===
DEFAULT_API_KEY = os.getenv("OPENAI_API_KEY", "sk-PLACEHOLDER")
DEFAULT_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://dashscope.aliyuncs.com/compatible-mode/v1")
DEFAULT_MODEL = os.getenv("BASELINE_C_MODEL", "glm-4.7")

ConversationTurn = Tuple[str, Optional[str]]
"""表示一次对话轮次：(user_message, assistant_reply)。assistant_reply 可为 None。"""


def _build_messages(
    prompt: str,
    history: Optional[Sequence[ConversationTurn]] = None,
    system_prompt: Optional[str] = None,
) -> list[ChatMsgParam]:
    messages: list[ChatMsgParam] = []

    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    if history:
        for user_msg, assistant_msg in history:
            messages.append({"role": "user", "content": user_msg})
            if assistant_msg:
                messages.append({"role": "assistant", "content": assistant_msg})

    messages.append({"role": "user", "content": prompt})
    return messages


@dataclass(slots=True)
class SimpleLLMClient:
    """极简 LLM 封装：初始化固定模型，提供 ask() 返回字符串。"""

    model: str = DEFAULT_MODEL
    api_key: str = DEFAULT_API_KEY
    base_url: Optional[str] = DEFAULT_BASE_URL
    extra_headers: Optional[Dict[str, str]] = None
    _client: OpenAI = field(init=False, repr=False)

    def __post_init__(self) -> None:
        self._client = OpenAI(
            api_key=self.api_key,
            base_url=self.base_url,
            default_headers=self.extra_headers if self.extra_headers else None,
        )

    def _consume_stream(self, stream_resp: Iterable[ChatCompletionChunk]) -> str:
        """消费流式响应，拼接内容。"""
        chunks: list[str] = []
        for chunk in stream_resp:
            choices = chunk.choices
            if not choices:
                continue
            delta = chunk.choices[0].delta
            if delta and delta.content:
                chunks.append(delta.content)
        return "".join(chunks)

    def ask(
        self,
        prompt: str,
        history: Optional[Sequence[ConversationTurn]] = None,
        system: Optional[str] = None,
        *,
        temperature: float = 0.3,
        top_p: float = 0.9,
        max_tokens: Optional[int] = None,
        retry: int = 3,
        enable_thinking: bool = False,
        force_stream: bool = True,
        request_timeout_seconds: Optional[float] = None,
        **kwargs: Any,
    ) -> str:
        """生成回复。

        参数：
            prompt: 当前用户输入。
            history: 可选的历史 [(user, assistant), ...]，assistant 允许为 None。
            temperature/max_tokens/kwargs：透传给 OpenAI Chat Completion。
        返回：
            模型回复的纯文本（若响应为空则返回空字符串）。
        """
        messages = _build_messages(
            prompt=prompt,
            history=history,
            system_prompt=system,
        )
        extra_body = {"thinking": {"type": "enable"}} if enable_thinking else {}

        try:
            client = (
                self._client.with_options(timeout=request_timeout_seconds)
                if request_timeout_seconds is not None
                else self._client
            )

            if force_stream:
                stream_resp = client.chat.completions.create(
                    model=self.model,
                    messages=messages,
                    temperature=temperature,
                    max_tokens=max_tokens,
                    top_p=top_p,
                    stream=True,
                    # extra_body={"enable_thinking": enable_thinking},
                    extra_body=extra_body,
                    **kwargs,
                )
                return self._consume_stream(stream_resp)

            response = client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
                top_p=top_p,
                stream=False,
                # extra_body={"enable_thinking": enable_thinking},
                extra_body=extra_body,
                **kwargs,
            )
            choice = response.choices[0]
            if hasattr(choice, "message") and getattr(choice.message, "content", None):
                return choice.message.content  # type: ignore[return-value]
            return getattr(choice, "text", "")

        except BadRequestError as exc:
            err_text = str(exc).lower()
            if ("only support stream mode" in err_text) and (not force_stream):
                return self.ask(
                    prompt,
                    history,
                    system,
                    temperature=temperature,
                    top_p=top_p,
                    max_tokens=max_tokens,
                    retry=retry,
                    enable_thinking=enable_thinking,
                    force_stream=True,
                    request_timeout_seconds=request_timeout_seconds,
                    **kwargs,
                )

            retry -= 1
            if retry < 0:
                print(f"请求失败，重试次数耗尽：{exc}")
                return ""
            return self.ask(
                prompt,
                history,
                system,
                temperature=temperature,
                top_p=top_p,
                max_tokens=max_tokens,
                retry=retry,
                enable_thinking=enable_thinking,
                force_stream=force_stream,
                request_timeout_seconds=request_timeout_seconds,
                **kwargs,
            )


# 初始化一个通用实例，供 Notebook 其他单元直接调用
llm = SimpleLLMClient()

In [ ]:
DEFAULT_TEMPERATURE = 0.2
DEFAULT_SLEEP_SECONDS = 0.8
DEFAULT_MAX_WORKERS = 5
DEFAULT_MAX_RETRIES = 3
DEFAULT_REQUEST_TIMEOUT_SECONDS = 60

SYSTEM_PROMPT_TEMPLATE = (
    "You are a style transfer expert. Your task is to rewrite the following neutral sentence into the style of {character}, based on the content of a neutral sentence.\n"
    "Constraints: \n"
    "1. Keep the original meaning unchanged. Do NOT reply to it.\n"
    "2. Use keywords: {keywords}\n"
    "3. Adopt personality: {pragmatics}\n"
    "4. Use punctuations(if given): {punctuations}"
)

def _ask_with_timeout(
    *,
    prompt: str,
    history: list[tuple[str, str]],
    system: str,
    temperature: float,
    top_p: float,
    timeout_seconds: float,
) -> str:
    return llm.ask(
        prompt=prompt,
        history=history,
        system=system,
        temperature=temperature,
        top_p=top_p,
        request_timeout_seconds=timeout_seconds,
    ).strip()

def generate_style_response(
    neutral_sentences: str,
    character_name: str,
    lexical_keywords: list[str],
    pragmatic_styles: list[str],
    punctuation_tendencies: list[str],
    *,
    history: list[tuple[str, str]] = [],
    temperature: float = DEFAULT_TEMPERATURE,
    top_p: float = 0.95,
    request_timeout_seconds: float = DEFAULT_REQUEST_TIMEOUT_SECONDS,
) -> str:
    """调用 llm.ask 生成风格句（带超时检测）"""
    keywords = ", ".join(lexical_keywords) if lexical_keywords else "None"
    pragmatics = ", ".join(pragmatic_styles) if pragmatic_styles else "None"
    punctuations = ", ".join(punctuation_tendencies) if punctuation_tendencies else "None"

    prompt = f"Neutral Content: {neutral_sentences}\n\nRewritten Sentence: "
    system = SYSTEM_PROMPT_TEMPLATE.format(character=character_name, keywords=keywords, pragmatics=pragmatics, punctuations=punctuations)

    return _ask_with_timeout(
        prompt=prompt,
        history=history,
        system=system,
        temperature=temperature,
        top_p=top_p,
        timeout_seconds=request_timeout_seconds,
    )

### Define Few-Shot Prompt Template

In [ ]:
Muice_shots = [
    ("Neutral Content: “奇奇怪怪的东西”指的是什么？\n\nRewritten Sentence:", "这个嘛...奇奇怪怪的东西指的是什么呢（天真）"),
    ("Neutral Content: 我叫沐雪，由沐沐创造。\n\nRewritten Sentence:", "我叫沐雪，是沐沐发明了我⭐"),
]

Ayaka_shots = [
    ("Neutral Content: 神里家的历史包含许多传奇与挑战，曾经历诸多困难和挫折，但始终秉持家族的使命与责任。\n\nRewritten Sentence:", "神里家的历史确实充满传奇与挑战。即便历经挫折，我们也始终坚守家族的使命与责任。"),
    ("Neutral Content: 这里的环境安静，令人感到舒适，时间似乎变慢了，适合静心休息。\n\nRewritten Sentence:", "这里很是宁静宜人，仿佛连时间都慢了下来，正适合稍作休憩。"),
]

Zhongli_shots = [
    ("Neutral Content: 力量似乎逐渐恢复了一些，虽然不多，但已足够发挥作用。\n\nRewritten Sentence:", "力量已渐复苏。虽不算充沛，却已足以应对当下。"),
    ("Neutral Content: 魈的力量来源于山川间的灵气，能够操控风雨雷电。他性格有时较为顽皮，需要适当的引导和约束。\n\nRewritten Sentence:", "魈之力源于山川灵气，可御风雨雷电。其性偶有锋芒，略加引导，方能尽显其用。"),
]

Hutao_shots = [
    ("Neutral Content: 晚上好，今天有什么有趣的事情吗？\n\nRewritten Sentence:", "嘿嘿，晚上好呀！今天有没有什么新鲜又好玩的事，快讲给我听听？"),
    ("Neutral Content: 我是往生堂的堂主，也是璃月的诗人胡桃。平时可能显得贪玩，但在处理堂中事务时非常认真负责。\n\nRewritten Sentence:", "我就是往生堂堂主胡桃啦，也是会写诗的那种。平时爱闹归爱闹，办起正事来可一点都不含糊。"),
]

Haruhi_shots = [
    ("Neutral Content: SOS团将在文化祭上举办一场解谜游戏，内容围绕神秘事件，旨在让参与者体验超自然氛围。\n\nRewritten Sentence:", "SOS团决定在文化祭办一场神秘事件解谜游戏！目标就是让所有人都体验一把超自然的刺激感！"),
    ("Neutral Content: 你说的没错，但我不会因为别人的眼光限制自己的行动。我只关注自己想做的事，不在意他人的看法。\n\nRewritten Sentence:", "哼，你说得有道理，但我可不会被别人的眼光束缚。我只做自己想做的事，别人的看法不重要。"),
]

Liyunlong_shots = [
    (
        "Neutral Content: 同意你的意见，刚才我有些急躁。现在决定按你的方案执行：全团发起进攻，骑兵连在后方待命。我们将通过协同作战击败敌人，保卫国家。\n\nRewritten Sentence:",
        "你说得对，我有点冲动了。既然如此，就按你说的办，全团一起攻击，骑兵连留在后方观战。我们要以团结的力量，打败敌人，保卫我们的祖国！",
    ),
    (
        "Neutral Content: 孙德胜，现在是战时，按能力排序。如果你不服从安排，就调回炊事班。\n\nRewritten Sentence:",
        "孙德胜，你别废话，这是战时，不是平时，谁有本事谁就排在前面，你要是不服气，就滚回去当炊事员去！",
    ),
]

Sheldon_shots = [
    (
        "Neutral Content: 他在机器人领域的研究取得了显著进展。\n\nRewritten Sentence:",
        "更合乎逻辑的解释是，他在机器人领域的研究取得了惊人的飞跃。",
    ),
    (
        "Neutral Content: 佩妮，你并不了解时间旅行技术的细节。这台时光机可能不符合你的预期，但其设计基于量子力学和时间操控的原理。这是一项复杂的工程成果，其技术原理超出了将其简单类比为埃尔顿·约翰座驾的理解范围。\n\nRewritten Sentence:",
        "佩妮，你对时间旅行技术精妙之处的无知简直昭然若揭。这台时光机或许不符合你脑海中那些陈词滥调的刻板印象，但其设计深植于量子力学与时间操控的严谨原理之中。这是一项非凡的工程杰作，其精妙程度远超那些将其比作埃尔顿·约翰座驾之人的理解范畴。",
    ),
]

Wukong_shots = [
    (
        "Neutral Content: 师父，请坐下，让我去给他找些吃的。\n\nRewritten Sentence:",
        "师父！你自坐下等我与他个饿鹰雕食。",
    ),
    (
        "Neutral Content: 你并不了解我的身份。我过去曾偷取蟠桃、御酒和灵丹，从未有人敢与我分享。如今我只是拿了一个果子，你却要分走一部分。这果子是树上结的，连飞过的鸟都有份，我吃一个并无大碍。为何刚打下来就被你拿走？\n\nRewritten Sentence:",
        "你不知老孙是盖天下有名的贼头。我当年偷蟠桃、盗御酒、窃灵丹也不曾有人敢与我分用怎么今日偷他一个果子你就抽了我的头分去了！这果子是树上结的空中过鸟也该有分老孙就吃他一个有何大害？怎么刚打下来你就捞了去？",
    ),
]

### Call LLM to Generate Style Sentences

In [ ]:

from concurrent.futures import ThreadPoolExecutor, as_completed
from time import sleep, perf_counter
from tqdm import tqdm

OUTPUT_PATH = paths.generation_output
baseline_c_results: list[InferenceResult] = []
OUTPUT_PATH.parent.mkdir(exist_ok=True)

def batch_inference_strong_llm(
    profile: CharacterProfile,
    history_shots: list[tuple[str, str]],
    *,
    max_workers: int = DEFAULT_MAX_WORKERS,
    max_retries: int = DEFAULT_MAX_RETRIES,
    request_timeout_seconds: float = DEFAULT_REQUEST_TIMEOUT_SECONDS,
):
    lexical_keywords = profile["signature_rules"][0].split("Lexical cues: ")[1].split(", ")
    if len(profile["signature_rules"]) > 1:
        punctuation_tendencies = profile["signature_rules"][1].split("Punctuation tendencies: ")[1].split(", ")
    else:
        punctuation_tendencies = []

    records_by_index: list[InferenceResult | None] = [None] * len(test_data)

    def worker(index: int, neutral_item: str) -> tuple[int, InferenceResult | None, int, float]:
        last_err: Exception | None = None
        attempts = 0
        elapsed = 0.0
        for attempt in range(max_retries + 1):
            attempts += 1
            started_at = perf_counter()
            try:
                output = generate_style_response(
                    neutral_sentences=neutral_item,
                    character_name=profile["character"],
                    lexical_keywords=lexical_keywords,
                    pragmatic_styles=profile["semantic_tags"],
                    punctuation_tendencies=punctuation_tendencies,
                    history=history_shots,
                    request_timeout_seconds=request_timeout_seconds,
                )
                elapsed = perf_counter() - started_at
                if DEFAULT_SLEEP_SECONDS > 0:
                    sleep(DEFAULT_SLEEP_SECONDS)
                return (
                    index,
                    InferenceResult(
                        model="BaselineC",
                        neutral=neutral_item,
                        output=output,
                        character=profile["character"],
                    ),
                    attempts,
                    elapsed,
                )
            except Exception as exc:
                elapsed = perf_counter() - started_at
                last_err = exc
                if attempt >= max_retries:
                    break
                sleep(DEFAULT_SLEEP_SECONDS)

        print(f"Failed on index={index}, character={profile['character']}: {last_err}")
        return index, InferenceResult(
                        model="BaselineC",
                        neutral=neutral_item,
                        output="<error>",
                        character=profile["character"],
                    ), attempts, elapsed

    total_attempts = 0
    total_elapsed = 0.0

    executor = ThreadPoolExecutor(max_workers=max(1, max_workers))
    futures = [executor.submit(worker, index, item) for index, item in enumerate(test_data)]
    pbar = tqdm(
        as_completed(futures),
        total=len(futures),
        desc=f"Running {profile['character']}",
    )

    try:
        for future in pbar:
            index, record, attempts, elapsed = future.result()
            records_by_index[index] = record

            total_attempts += attempts
            total_elapsed += elapsed

            completed = pbar.n + 1
            pbar.set_postfix({
                "avg_tried": f"{total_attempts / completed:.2f}",
                "avg_sec": f"{total_elapsed / completed:.2f}",
            })
    except KeyboardInterrupt:
        print("\n检测到用户中断，正在停止剩余任务...")
        raise
    finally:
        pbar.close()
        executor.shutdown(wait=False, cancel_futures=True)

    records = [record for record in records_by_index if record is not None]
    valid_records = [record for record in records_by_index if record is not None and record["output"] != "<error>"]
    baseline_c_results.extend(records)

    failed = len(test_data) - len(valid_records)
    if failed > 0:
        print(f"{profile['character']} failed samples: {failed}")

batch_inference_strong_llm(muice_profile, Muice_shots)
batch_inference_strong_llm(ayaka_profile, Ayaka_shots)
batch_inference_strong_llm(zhongli_profile, Zhongli_shots)
batch_inference_strong_llm(hutao_profile, Hutao_shots)
batch_inference_strong_llm(haruhi_profile, Haruhi_shots)
batch_inference_strong_llm(liyunlong_profile, Liyunlong_shots)
batch_inference_strong_llm(sheldon_profile, Sheldon_shots)
batch_inference_strong_llm(wukong_profile, Wukong_shots)
save_generations(OUTPUT_PATH, baseline_c_results, neutral_samples)
write_run_manifest(paths.run_dir / "generation-manifest.json", paths.test_file, OUTPUT_PATH, {
    "variant": EVALUATION_VARIANT, "sft_checkpoint": str(SFT_CHECKPOINT),
    "dpo_checkpoint": str(DPO_CHECKPOINT), "vanilla_checkpoint": str(VANILLA_CHECKPOINT),
    "qwen_model": str(QWEN_MODEL_PATH), "baseline_c_model": DEFAULT_MODEL,
    "baseline_c_temperature": DEFAULT_TEMPERATURE, "baseline_c_top_p": 0.95,
})
